# 02 - CasADi Function Objects

This notebook introduces `ca.Function`, based on CasADi documentation Section 4. Function objects are the bridge between symbolic expressions and reusable numerical computations.

Learning goals:

- Create functions with named and unnamed inputs/outputs.
- Evaluate functions with Python lists, NumPy arrays, and `DM` values.
- Package residuals, costs, Jacobians, gradients, and Hessians.
- Compose `SX` functions inside larger `MX` graphs.

In [ ]:
# Colab setup.
# These tutorials assume a fresh Google Colab runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "casadi",
    "numpy",
    "matplotlib",

])


In [ ]:
import casadi as ca
import numpy as np

print("CasADi version:", ca.__version__)

## Creating A Function

A `Function` has a name, a list of symbolic inputs, and a list of symbolic outputs.

In [ ]:
x = ca.SX.sym("x", 2)
y = ca.SX.sym("y")

f = ca.Function("f", [x, y], [x, ca.sin(y) * x])
print(f)
print("positional call:", f([0.1, 0.2], 2.0))

Named inputs and outputs make function calls much easier to read once examples become larger.

In [ ]:
f_named = ca.Function(
    "f_named",
    [x, y],
    [x, ca.sin(y) * x],
    ["x", "y"],
    ["copy_of_x", "scaled_x"]
)

out = f_named(x=ca.DM([0.1, 0.2]), y=2.0)
print("output keys:", out.keys())
print("copy_of_x =", out["copy_of_x"])
print("scaled_x =", out["scaled_x"])

## Multiple Outputs For Analysis Tools

One function can expose the quantities you want to inspect: residuals, costs, Jacobians, gradients, and Hessians.

In [ ]:
u = ca.SX.sym("u", 2)
r = ca.vertcat(
    u[0] + 2 * u[1] - 1,
    ca.sin(u[0]) - u[1],
    u[0]**2 + u[1] - 0.5
)
cost = 0.5 * ca.sumsqr(r)
J = ca.jacobian(r, u)
grad = ca.gradient(cost, u)
H, _ = ca.hessian(cost, u)

tools = ca.Function(
    "least_squares_tools",
    [u],
    [r, cost, J, grad, H],
    ["u"],
    ["r", "cost", "J", "grad", "H"]
)

out = tools(u=np.array([0.2, -0.4]))
for name, value in out.items():
    print(name, "=\n", value)

## Function Inputs Are Numeric At Evaluation Time

The same function can be called with lists, NumPy arrays, or `DM` values.

In [ ]:
for value in ([0.0, 0.0], np.array([0.2, -0.4]), ca.DM([1.0, 0.5])):
    result = tools(value)
    print("input type:", type(value).__name__, "cost:", float(result[1]))

## Composing SX Functions In MX Graphs

A common pattern is to define compact scalar or vector expressions in `SX`, package them as a `Function`, and then call that function inside an `MX` graph.

In [ ]:
z_sx = ca.SX.sym("z", 2)
small_expr = ca.vertcat(
    z_sx[0]**2 + z_sx[1],
    ca.cos(z_sx[0] - z_sx[1])
)
small_fun = ca.Function("small_fun", [z_sx], [small_expr])

z_mx = ca.MX.sym("z", 2)
large_expr = ca.vertcat(small_fun(z_mx), z_mx[0] + z_mx[1])
large_fun = ca.Function("large_fun", [z_mx], [large_expr])

print("large MX expression:", large_expr)
print("large_fun([0.3, -0.2]) =", large_fun([0.3, -0.2]))

## Reusable Cost Function Package

The next notebooks use this pattern repeatedly: build expressions once, then evaluate the packaged functions many times.

In [ ]:
def make_line_fit_functions(x_data, y_data):
    theta = ca.SX.sym("theta", 2)
    prediction = theta[0] * ca.DM(x_data) + theta[1]
    residual = prediction - ca.DM(y_data)
    cost = 0.5 * ca.sumsqr(residual)
    J = ca.jacobian(residual, theta)
    grad = ca.gradient(cost, theta)
    H, _ = ca.hessian(cost, theta)

    return ca.Function(
        "line_fit_package",
        [theta],
        [prediction, residual, cost, J, grad, H],
        ["theta"],
        ["prediction", "residual", "cost", "J", "grad", "H"]
    )

line_fit = make_line_fit_functions([0, 1, 2, 3], [1.0, 1.8, 3.2, 3.9])
result = line_fit(theta=[1.0, 0.5])
for name, value in result.items():
    print(name, "=\n", value)

## Exercise

Create a function that packages the residual, cost, Jacobian, gradient, and Hessian for the following nonlinear residual.

Let

$$
q = \begin{bmatrix} q_0 \\ q_1 \end{bmatrix}.
$$

Define

$$
r(q) =
\begin{bmatrix}
q_0 + q_1 - 1 \\
\sin(q_0) - 0.5 \\
q_1^2 - q_0
\end{bmatrix}.
$$

The scalar least-squares objective is

$$
\begin{aligned}
\min_{q \in \mathbb{R}^2} \quad
& \frac{1}{2} \|r(q)\|_2^2.
\end{aligned}
$$

Package these expressions in a `Function` named `residual_cost_derivative_fun`:

$$
J_r(q) = \frac{\partial r}{\partial q}, \qquad
\nabla f(q) = \frac{\partial f}{\partial q}, \qquad
\nabla^2 f(q) = \frac{\partial^2 f}{\partial q^2}.
$$